# COMP - 2040: Final Project  
**Name:** Troy Dela Rosa  
**SID#** 0213352  
**Date** April 11, 2026  
**Instructor:** Chris Mac  

## 1. Introduction

### How Instructor Feedback Changed This Project

In weeks 12 and 13 I focused on downtown Winnipeg business license activity from 2023 to 2025. The data came from the City of Winnipeg open-data portal, and the plan was to look at whether license patterns showed a measurable decline.

After the week 13 check-in, Chris pointed out that a three-year window is too narrow to tell whether what I was seeing was a real structural trend or just short-term noise. He suggested pushing the timeline back as far as the data would allow.

That feedback changed the project in a big way. I called 311 and submitted a formal data request through the City's website, but the historical license records I was hoping for were not available in time. Instead I expanded the analysis by building and collecting additional datasets from public sources — CBRE vacancy reports, Downtown BIZ snapshots, StatCan census data, and local news reporting — to piece together a longer-term picture stretching back to 2010.

The result is a project that no longer just asks *"is downtown declining?"* but instead tries to understand *how, when, and in what way* downtown Winnipeg has been changing — and whether the $2.3 billion in tracked investment represents recovery, or something structurally different.

### Central Argument

Downtown Winnipeg is not recovering in the traditional sense. It is converting from a retail-and-office-anchored model to a residential and institutional one. The investment does not attempt to restore what was lost — it bets on a different urban form.

### Three Analytical Questions

1. **How has the balance of Growth vs. Transition events shifted across the three phases of downtown development?**
2. **Has downtown business activity declined — and which sectors were hit hardest?**
3. **Can we predict the delivery status of a housing project (Completed / Under Construction / Planned) based on its characteristics?**

## 2. Data Sources and Research Design

### What datasets am I working with?

This project uses several datasets I built and collected from public sources:

- **`downtown_wpg_gantt_sourced_2026.csv`** — 28 major structural events in Downtown Winnipeg between 2010 and 2026, including construction projects, retail closures, and policy decisions. Each row records a start date, end date, event category, and a source provenance rating.
- **`housing_pipeline_2026.csv`** — residential development projects in the downtown pipeline, with unit count ranges, delivery status, and confidence levels.
- **`Business_Licenses_20260410.csv`** — City of Winnipeg open data: downtown business license records from 2021 to 2026.
- **`Winnipeg_Synthetic_License_Model_2010_2024.xlsx`** — A calibrated synthetic model that estimates downtown business activity from 2010 to 2020, where official open data is unavailable.
- **`downtown_wpg_sources_2026.csv`** — A source provenance registry documenting every figure with its source, confidence score, and usage type.

### Where does the data come from?

The business license data comes directly from the City of Winnipeg Open Data portal. The structural events and housing pipeline datasets were built by me from public sources: CBRE market reports, Downtown Winnipeg BIZ annual snapshots, and CBC Manitoba news reporting. The full source registry documents 53 sources.

### Data Layer Structure

Each section is labeled so the reader knows how reliable every number is:

| Layer | Label | Meaning |
|---|---|---|
| **L1 — Observed** | `[OBSERVED]` | Directly confirmed from primary sources |
| **L2 — Reconstructed** | `[RECONSTRUCTED]` | Calibrated estimates filling known data gaps |
| **L3 — Inferred** | `[INFERRED]` | Patterns derived from L1/L2, not direct measurements |
| **L4 — Scenario** | `[SCENARIO]` | Model outputs under stated assumptions |

## 3. Setup

In [ ]:
# standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# scikit-learn imports for prediction section
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# dark chart style — all charts inherit this
plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor':   '#161921',
    'axes.edgecolor':   '#2a2e3a',
    'axes.labelcolor':  '#e0ddd5',
    'text.color':       '#e0ddd5',
    'xtick.color':      '#8a8780',
    'ytick.color':      '#8a8780',
    'grid.color':       '#2a2e3a',
    'font.family':      'sans-serif',
    'font.size':        11,
    'figure.dpi':       120
})

# colour constants
ACCENT    = '#c9a44a'   # gold
GROWTH_C  = '#4e9a6e'   # green
TRANS_C   = '#5a7bbf'   # blue
DECLINE_C = '#bf4e4e'   # red

PHASE_COLORS = {
    'Pre-2015 establishment':    '#6e8a9e',
    '2015\u20132020 transition': DECLINE_C,
    '2020+ restructuring':       GROWTH_C,
}

TYPE_COLORS = {
    'Growth':         '#4e9a6e',
    'Infrastructure': '#5a7bbf',
    'Transition':     '#bf4e4e',
    'Adaptive Reuse': '#c9a44a',
    'Policy':         '#8a6e9a',
}

print("Ready.")

### Helper Module — New Function for Week 14

Weeks 12–13 already had three helper functions in `src/helpers.py`:
- `clean_column_names()` — lowercase + underscore formatting
- `create_is_closed()` — flags closed/cancelled/ceased licenses
- `filter_downtown()` — keeps only downtown Community Characterization Area

For week 14 I added a fourth function that assigns each gantt event to its development phase based on start year. I'm defining it here so the notebook is self-contained, but it also lives in `helpers.py`.